# 🧠 Memlayer Self-Hosting Demo

This notebook demonstrates:
1. ✅ Using Memlayer with custom salience configurations (library mode)
2. ✅ Testing the self-hosted API endpoints
3. ✅ Multi-user isolation
4. ✅ Creating and testing custom scoring functions

## Prerequisites

- OpenAI API key
- (Optional) Self-hosted Memlayer API running

## Setup

In [ ]:
# Install Memlayer
!pip install -q git+https://github.com/thebnbrkr/memlayer.git

In [ ]:
# Import libraries
import os
from getpass import getpass

# Set your OpenAI API key
if 'OPENAI_API_KEY' not in os.environ:
    os.environ['OPENAI_API_KEY'] = getpass('Enter your OpenAI API key: ')

## Part 1: Library Mode - Custom Salience Configuration

Use Memlayer directly in your Python code with custom salience rules.

In [ ]:
from memlayer import OpenAI
from memlayer.config.salience import (
    TenantSalienceConfig,
    SalienceComponent,
    ScoringFunctionType,
    AdaptiveThresholdConfig,
    ThresholdStrategy,
    DecisionRule
)

# Create a custom salience config - only store technical/coding facts
tech_config = TenantSalienceConfig(
    tenant_id="demo_user",
    config_name="tech_only",
    components=[
        SalienceComponent(
            name="technical_keywords",
            weight=0.6,
            scoring_function=ScoringFunctionType.KEYWORD_MATCH,
            scoring_config={
                "keywords": [
                    "python", "code", "programming", "algorithm", "AI", 
                    "machine learning", "software", "API", "database", "docker"
                ],
                "case_sensitive": False
            }
        ),
        SalienceComponent(
            name="length_bonus",
            weight=0.4,
            scoring_function=ScoringFunctionType.LENGTH_BONUS,
            scoring_config={
                "min_length": 20,
                "max_length": 200,
                "optimal_length": 100
            }
        )
    ],
    threshold_config=AdaptiveThresholdConfig(
        strategy=ThresholdStrategy.ABSOLUTE,
        absolute_threshold=0.5  # Only store if score >= 0.5
    ),
    decision_rules=[
        DecisionRule(
            name="always_store_high_tech",
            priority=1,
            condition="technical_keywords > 0.8",
            action="STORE",
            reasoning="High technical relevance - always store"
        )
    ]
)

print("✅ Tech-focused salience config created!")
print(f"   Config: {tech_config.config_name}")
print(f"   Components: {[c.name for c in tech_config.components]}")
print(f"   Threshold: {tech_config.threshold_config.strategy} = {tech_config.threshold_config.absolute_threshold}")

In [ ]:
# Initialize Memlayer client with custom config
client = OpenAI(
    model="gpt-4o-mini",
    user_id="alice",
    tenant_id="demo_user",
    storage_path="./demo_storage",
    operation_mode="online",
    salience_config=tech_config  # 🔥 Custom config!
)

print("✅ Memlayer client initialized with custom salience config!")

### Test 1: Technical Fact (Should Store)

In [ ]:
# Send a technical message
response = client.chat([
    {"role": "user", "content": "I'm learning Python to build machine learning APIs with Docker containers."}
])

print("🤖 Response:", response)
print()

# Check salience logs
logs = client.get_salience_logs()
if logs:
    last_log = logs[-1]
    print("📊 Salience Analysis:")
    print(f"   Score: {last_log['score']:.3f}")
    print(f"   Decision: {last_log['decision']}")
    print(f"   Component Scores: {last_log['component_scores']}")
    print(f"   Threshold: {last_log['threshold']:.3f}")
    print(f"   Reasoning: {last_log['reasoning']}")
else:
    print("⚠️ No salience logs yet")

### Test 2: Non-Technical Fact (Should Skip)

In [ ]:
# Send a non-technical message
response = client.chat([
    {"role": "user", "content": "I had pizza for lunch today."}
])

print("🤖 Response:", response)
print()

# Check salience logs
logs = client.get_salience_logs()
if logs:
    last_log = logs[-1]
    print("📊 Salience Analysis:")
    print(f"   Score: {last_log['score']:.3f}")
    print(f"   Decision: {last_log['decision']} ← Should be SKIP!")
    print(f"   Component Scores: {last_log['component_scores']}")
    print(f"   Threshold: {last_log['threshold']:.3f}")
    print(f"   Reasoning: {last_log['reasoning']}")

### Test 3: View All Stored Memories

In [ ]:
# View all salience decisions
print("📋 All Salience Decisions:\n")
for i, log in enumerate(client.get_salience_logs(), 1):
    print(f"{i}. {log['decision']}: {log['fact'][:60]}...")
    print(f"   Score: {log['score']:.3f} (threshold: {log['threshold']:.3f})")
    print(f"   Components: {log['component_scores']}")
    print()

## Part 2: Different Salience Profiles

Create different configs for different use cases.

In [ ]:
# Emotional/Personal Config - stores feelings and relationships
emotional_config = TenantSalienceConfig(
    tenant_id="demo_user",
    config_name="emotional_memory",
    components=[
        SalienceComponent(
            name="emotional_keywords",
            weight=1.0,
            scoring_function=ScoringFunctionType.KEYWORD_MATCH,
            scoring_config={
                "keywords": [
                    "love", "hate", "happy", "sad", "angry", "excited",
                    "friend", "family", "relationship", "feel", "emotion"
                ],
                "case_sensitive": False
            }
        )
    ],
    threshold_config=AdaptiveThresholdConfig(
        strategy=ThresholdStrategy.ABSOLUTE,
        absolute_threshold=0.3  # Lower threshold - store more
    )
)

# Create client with emotional config
emotional_client = OpenAI(
    model="gpt-4o-mini",
    user_id="bob",
    tenant_id="demo_user",
    storage_path="./demo_storage",
    salience_config=emotional_config
)

print("✅ Emotional memory config created!")

In [ ]:
# Test emotional config
response = emotional_client.chat([
    {"role": "user", "content": "I'm so happy! My best friend is visiting this weekend."}
])

print("🤖 Response:", response)
print()

logs = emotional_client.get_salience_logs()
if logs:
    last_log = logs[-1]
    print("📊 Emotional Salience:")
    print(f"   Score: {last_log['score']:.3f}")
    print(f"   Decision: {last_log['decision']} ← Should be STORE!")
    print(f"   Matched keywords: {last_log['component_scores']}")

## Part 3: Advanced - Boolean Logic Rules

Use decision rules with complex boolean conditions.

In [ ]:
# Config with multiple components and decision rules
advanced_config = TenantSalienceConfig(
    tenant_id="demo_user",
    config_name="advanced_rules",
    components=[
        SalienceComponent(
            name="tech",
            weight=0.5,
            scoring_function=ScoringFunctionType.KEYWORD_MATCH,
            scoring_config={
                "keywords": ["code", "python", "AI", "software"]
            }
        ),
        SalienceComponent(
            name="personal",
            weight=0.5,
            scoring_function=ScoringFunctionType.KEYWORD_MATCH,
            scoring_config={
                "keywords": ["I", "my", "me", "mine"]
            }
        )
    ],
    threshold_config=AdaptiveThresholdConfig(
        strategy=ThresholdStrategy.ABSOLUTE,
        absolute_threshold=0.5
    ),
    decision_rules=[
        DecisionRule(
            name="store_if_both_high",
            priority=1,
            condition="tech > 0.7 and personal > 0.7",  # Boolean logic!
            action="STORE",
            reasoning="High tech + personal relevance"
        ),
        DecisionRule(
            name="skip_if_only_personal",
            priority=2,
            condition="personal > 0.8 and tech < 0.2",
            action="SKIP",
            reasoning="Too personal, not technical enough"
        )
    ]
)

print("✅ Advanced config with boolean rules created!")
print(f"   Rules: {[r.name for r in advanced_config.decision_rules]}")

## Part 4: Multi-User Isolation

Test that different users have separate memory stores.

In [ ]:
# Create two clients for different users
alice = OpenAI(
    model="gpt-4o-mini",
    user_id="alice",
    tenant_id="friend_group",
    storage_path="./demo_storage",
    salience_config=tech_config
)

bob = OpenAI(
    model="gpt-4o-mini",
    user_id="bob",
    tenant_id="friend_group",
    storage_path="./demo_storage",
    salience_config=tech_config
)

print("✅ Created clients for Alice and Bob")

In [ ]:
# Alice shares info about herself
alice.chat([{"role": "user", "content": "I work at Google as a Python developer."}])
print("Alice: Shared info about working at Google")

# Bob shares different info
bob.chat([{"role": "user", "content": "I work at Meta as a machine learning engineer."}])
print("Bob: Shared info about working at Meta")
print()

# Check Alice's logs
print("📊 Alice's memories:")
for log in alice.get_salience_logs():
    if log['decision'] == 'STORE':
        print(f"   - {log['fact'][:60]}...")

print()

# Check Bob's logs
print("📊 Bob's memories:")
for log in bob.get_salience_logs():
    if log['decision'] == 'STORE':
        print(f"   - {log['fact'][:60]}...")

print()
print("✅ Memories are isolated by user_id!")

## Part 5: Testing Self-Hosted API (Optional)

If you have Memlayer API running via Docker, test the REST endpoints.

In [ ]:
import requests
import json

# Set your API URL (change if not running locally)
API_URL = "http://localhost:8000"

# Test health endpoint
try:
    response = requests.get(f"{API_URL}/health")
    if response.status_code == 200:
        print("✅ API is healthy!")
        print(json.dumps(response.json(), indent=2))
        api_available = True
    else:
        print(f"⚠️ API returned status {response.status_code}")
        api_available = False
except Exception as e:
    print(f"❌ API not available: {e}")
    print("   (This is OK if you're not running Docker. Skip to Part 6)")
    api_available = False

In [ ]:
# Create a config via API
if api_available:
    config_data = {
        "tenant_id": "api_test",
        "config_name": "api_created_config",
        "components": [
            {
                "name": "keywords",
                "weight": 1.0,
                "scoring_function": "keyword_match",
                "scoring_config": {
                    "keywords": ["important", "urgent", "critical"]
                }
            }
        ],
        "threshold_config": {
            "strategy": "absolute",
            "absolute_threshold": 0.5
        },
        "decision_rules": []
    }
    
    response = requests.post(
        f"{API_URL}/api/config/salience/",
        json=config_data
    )
    
    if response.status_code == 201:
        print("✅ Config created via API!")
        print(json.dumps(response.json(), indent=2))
    else:
        print(f"❌ Failed to create config: {response.text}")

In [ ]:
# Test the config
if api_available:
    test_data = {
        "fact": "This is an urgent and important message about critical systems."
    }
    
    response = requests.post(
        f"{API_URL}/api/config/salience/api_created_config/test?tenant_id=api_test",
        json=test_data
    )
    
    if response.status_code == 200:
        result = response.json()
        print("✅ Test result:")
        print(f"   Salience Score: {result['salience_score']:.3f}")
        print(f"   Threshold: {result['threshold']:.3f}")
        print(f"   Decision: {result['decision']}")
        print(f"   Component Scores: {result['component_scores']}")
    else:
        print(f"❌ Test failed: {response.text}")

## Part 6: Summary

What we tested:

✅ **Custom Salience Configs**: Created tech-focused, emotional, and advanced configs  
✅ **Multiple Scoring Functions**: Keyword matching, length bonus  
✅ **Decision Rules**: Boolean logic with priority evaluation  
✅ **Multi-User Isolation**: Alice and Bob have separate memories  
✅ **API Testing**: CRUD operations and testing endpoints (if Docker running)  

## Next Steps

1. **Deploy with Docker**: Follow `DOCKER_QUICKSTART.md`
2. **Share with friends**: Give them different `user_id` values
3. **Create custom configs**: Experiment with different scoring functions
4. **Monitor usage**: Check `/health` and audit logs

## Cost Estimate

For you + 5 friends:
- **Hosting**: ~$12-15/month (DigitalOcean, Hetzner)
- **Storage**: ~$3/month
- **Total**: ~$15-18/month ($2.50-3 per person)

See `SELF_HOSTING.md` for complete deployment guide!